In [1]:
!pip install langchain langchain-groq langchain-community langchain-core transformers sentence-transformers faiss-cpu python-dotenv torch

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
token = os.getenv("GROQ_API_KEY")
print("Token loaded:", "✅" if token else "❌ Not found")

Token loaded: ✅


In [4]:
import sys
!{sys.executable} -m pip install langchain-groq

  Using cached langchain_groq-1.1.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached langchain_groq-1.1.2-py3-none-any.whl (19 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)

   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 2/2 [langchain-groq]



# Basic LLm Call

In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens = 200
)
def ask_llm(query):
    response = llm.invoke(query)
    # response is AIMessage — need .content for plain text
    return response.content

print(ask_llm("Explain backpropagation in simple terms"))

**Backpropagation in Simple Terms**

Backpropagation is a key concept in machine learning, particularly in neural networks. It's a way to train a network by adjusting the connections between its layers to minimize errors.

**Imagine a Simple Example**

Think of a neural network as a series of layers, each with its own set of "neurons" (or nodes). Each neuron receives input from the previous layer, performs a calculation, and sends the output to the next layer.

**The Goal**

The goal of backpropagation is to adjust the connections between the neurons so that the output of the network matches the desired output (or target).

**How it Works**

Here's a step-by-step explanation:

1. **Forward Pass**: The network processes the input data, and the output is calculated.
2. **Error Calculation**: The difference between the calculated output and the desired output is calculated (this is called the error).
3. **Backward Pass**: The error is propagated backwards through the


# PromptTemplate usage

In [18]:
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate

# a) Basic PromptTemplate
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)

print("a) Basic PromptTemplate \n",template.format(topic="Neural Networks"))
# Output: Explain Neural Networks in simple terms

# b) ChatPromptTemplate with system message
chat_prompt = ChatPromptTemplate([
    ("system", "You are an expert AI educator"),
    ("user" , "{question}")
])
chain = chat_prompt | llm
response = chain.invoke({"question" : "What is backpropagation?"})
print("\nb) ChatPromptTemplate with system message \n",response.content)

a) Basic PromptTemplate 
 Explain Neural Networks in simple terms

b) ChatPromptTemplate with system message 
 Backpropagation is a fundamental concept in machine learning, particularly in the field of neural networks. It's a supervised learning algorithm used to train artificial neural networks to learn from data.

**What does "backpropagation" mean?**

The term "backpropagation" comes from the idea of propagating errors backwards through the network. In other words, it's a method for computing the gradient of the loss function with respect to the model's parameters, which is essential for updating the weights and biases of the network during training.

**How does backpropagation work?**

Here's a step-by-step explanation:

1. **Forward pass**: The input data is fed into the network, and the output is computed by propagating the input through the network layer by layer.
2. **Error computation**: The difference between the predicted output and the actual output (target) is computed, wh

# Chain  Tools  Memory   

In [25]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful tutor."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Chain
chain = prompt | llm | StrOutputParser()

# Session store
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Add memory
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Config
config = {"configurable": {"session_id": "demo"}}

# Turn 1
r1 = chain_with_memory.invoke(
    {"input": "I struggle with list comprehensions."},
    config=config
)
print("Turn 1:", r1)

# Turn 2 (memory works)
r2 = chain_with_memory.invoke(
    {"input": "Give me a beginner exercise for that."},
    config=config
)
print("Turn 2:", r2)

Turn 1: List comprehensions can be a bit tricky at first, but they're a powerful tool in Python. I'd be happy to help you understand them better.

**What is a list comprehension?**

A list comprehension is a concise way to create a new list from an existing list or other iterable. It's a one-liner that combines the for loop and if statement to create a new list.

**Basic syntax**

The basic syntax of a list comprehension is:
```python
new_list = [expression for element in iterable if condition]
```
Let's break it down:

* `expression` is the operation you want to perform on each element.
* `element` is the variable that takes on the value of each element in the iterable.
* `iterable` is the list or other iterable you're working with.
* `condition` is an optional filter that allows you to include only certain elements.

**Example 1: Simple list comprehension**

Suppose we have a list of numbers and
Turn 2: Here's a beginner exercise to help you practice list comprehensions:

**Exercise: